# TorchRL logging progress

In this tutorial, we will learn how to use log and visualize our agents progress

* Note: this notebook was modified from the official tutorials provided by torchRL, you can check it out here: https://docs.pytorch.org/rl/stable/tutorials

### Install TorchRL

First let's install torchrl and tensordict.


In [1]:
!pip install tensordict
!pip install torchrl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.8/536.8 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.2 MB/s eta 0:00:00


In [2]:
#Option 1: easy
#If you are running this from colab you can run:
#!pip install tensordict
#!pip install torchrl

#Option 2: harder
#if you are using your own computer you should create a virtual environment, activate it, and then use pip to install the needed libraries
#conda create -n torchrl-env python=3.11 -y
#conda activate torchrl-env
#pip install torchrl tensordict torch torchvision 'gymnasium[all]'
#Note: for GPU support this might be a little different, and might vary by device!

Load some libraries

In [3]:
import torchvision

from tensordict.nn import TensorDictSequential
from torchrl.modules import EGreedyModule

import torch
from tensordict.nn import TensorDictModule
from tensordict import TensorDict
from torchrl.envs import GymEnv
from torch import nn

from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import QValueActor
from torch.optim import Adam
from torchrl.modules import QValueActor
from torchrl.objectives import DQNLoss


### Logging

Logging is crucial for reporting your results to the outside world and for you to check that your algorithm is learning properly. TorchRL has several loggers that interface with custom backends such as wandb (WandbLogger), tensorboard (TensorBoardLogger) or a lightweight and portable CSV logger (CSVLogger) that you can use pretty much everywhere.

Loggers are located in the torchrl.record module and the various classes can be found in the API reference.

Building a logger requires at least an experiment name and possibly a logging directory and other hyperparameters.

In [4]:
from torchrl.record import CSVLogger

#create a logger
logger = CSVLogger(exp_name="my_exp")

After creating the logger you should see a new folder with the experiment name, and some sub folders.

Once the logger is instantiated, the only thing left to do is call the logging methods! For example, log_scalar() is used in several places across the training examples to log values such as reward, loss value or time elapsed for executing a piece of code.

Let's log a scalar value.

In [5]:
#log a scalar value
logger.log_scalar("my_scalar", 0.4)

You should now see this show up in the scalars folder as a csv file.

Let's see next how we can integrate this into our DQN loop from the last tutorial

### Logging training stats

Add some more here on using the logger (or wait to build it into the DQN?)

### Recording videos


Finally, it can come in handy to record videos of a simulator. Some environments (e.g., Atari games) are already rendered as images whereas others require you to create them as such. Fortunately, in most common cases, rendering and recording videos isn’t too difficult.

Let’s first see how we can create a Gym environment that outputs images alongside its observations. GymEnv accept two keywords for this purpose:

* from_pixels=True will make the env step function write a "pixels" entry containing the images corresponding to your observations,

* pixels_only=False will indicate that you want the observations to be returned as well.



In [8]:
from torchrl.envs import GymEnv

env = GymEnv("CartPole-v1", from_pixels=True, pixels_only=False)

print(env.rollout(max_steps=3))

from torchrl.envs import TransformedEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: De

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([3, 4]), device=cpu, dtype=torch.float32, is_shared=False),
                pixels: Tensor(shape=torch.Size([3, 400, 600, 3]), device=cpu, dtype=torch.uint8, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([3]),
            device=None,
            

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


You should see the "pixels" entry in the TensorDict.

We now have built an environment that renders images with its observations. To record videos, we will need to combine that environment with a recorder and the logger (the logger providing the backend to save the video). This will happen within a transformed environment, like the one we saw in the first tutorial.



In [9]:
from torchrl.record import VideoRecorder

recorder = VideoRecorder(logger, tag="my_video", video_format="mp4")
record_env = TransformedEnv(env, recorder)

rollout = record_env.rollout(max_steps=3)

# Uncomment this line to save the video on disk:
recorder.dump()

When running this environment, all the "pixels" entries will be saved in a local buffer (i.e. RAM) and dumped in a video on demand (to prevent excessive RAM usage, you are advised to call this method whenever appropriate!):



In [10]:
rollout = record_env.rollout(max_steps=3)

# Uncomment this line to save the video on disk:
recorder.dump()

In [11]:
from torchrl.record import VideoRecorder
from torchrl.record.loggers import CSVLogger
import torchrl

# CSVLogger is the simplest way to get actual video files
logger = CSVLogger(exp_name="my_experiment", log_dir="./logs", video_format="mp4")

recorder = VideoRecorder(logger, tag="my_video")
record_env = TransformedEnv(env, recorder)

rollout = record_env.rollout(max_steps=200)
recorder.dump()

In this specific case, the video format can be chosen when instantiating the CSVLogger.

(If you want to customise how your video is recorded, have a look at our knowledge base.)

This is all we wanted to cover in the getting started tutorial. You should now be ready to code your first training loop with TorchRL!



### Putting it all together

Time to wrap up everything we’ve learned so far in this Getting Started series!

In this tutorial, we will be writing the most basic training loop there is using only components we have presented in the previous lessons.

We’ll be using DQN with a CartPole environment as a prototypical example.

We will be voluntarily keeping the verbosity to its minimum, only linking each section to the related tutorial.

#### Building the environment

We’ll be using a gym environment with a StepCounter transform. If you need a refresher, check our these features are presented in the environment tutorial.



In [12]:
import torch
from torchrl.envs import GymEnv, StepCounter, TransformedEnv
from tensordict.nn import TensorDictModule as Mod, TensorDictSequential as Seq
import time


torch.manual_seed(0)

env = TransformedEnv(GymEnv("CartPole-v1"), StepCounter())
env.set_seed(0)



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


795726461

#### Designing a policy

The next step is to build our policy. We’ll be making a regular, deterministic version of the actor to be used within the loss module and during evaluation. Next, we will augment it with an exploration module for inference.



In [13]:
from torchrl.modules import EGreedyModule, MLP, QValueModule

value_mlp = MLP(out_features=env.action_spec.shape[-1], num_cells=[64, 64])

value_net = Mod(value_mlp, in_keys=["observation"], out_keys=["action_value"])

policy = Seq(value_net, QValueModule(spec=env.action_spec))
exploration_module = EGreedyModule(
    env.action_spec, annealing_num_steps=100_000, eps_init=0.5
)

policy_explore = Seq(policy, exploration_module)

#### Data Collector and replay buffer

Here comes the data part: we need a data collector to easily get batches of data and a replay buffer to store that data for training.



In [14]:
from torchrl.collectors import SyncDataCollector
from torchrl.data import LazyTensorStorage, ReplayBuffer

init_rand_steps = 5000
frames_per_batch = 100
optim_steps = 10

collector = SyncDataCollector(
    env,
    policy_explore,
    frames_per_batch=frames_per_batch,
    total_frames=-1,
    init_random_frames=init_rand_steps,
)

rb = ReplayBuffer(storage=LazyTensorStorage(100_000))



<frozen abc>:106: DeprecationWarning: Creating VanillaWeightUpdater which inherits from WeightUpdaterBase is deprecated. Please use WeightSyncScheme from torchrl.weight_update.weight_sync_schemes instead. This will be removed in a future version.
<frozen abc>:106: DeprecationWarning: Creating MultiProcessedWeightUpdater which inherits from WeightUpdaterBase is deprecated. Please use WeightSyncScheme from torchrl.weight_update.weight_sync_schemes instead. This will be removed in a future version.
<frozen abc>:106: DeprecationWarning: Creating RemoteModuleWeightUpdater which inherits from WeightUpdaterBase is deprecated. Please use WeightSyncScheme from torchrl.weight_update.weight_sync_schemes instead. This will be removed in a future version.
<frozen abc>:106: DeprecationWarning: Creating RayWeightUpdater which inherits from WeightUpdaterBase is deprecated. Please use WeightSyncScheme from torchrl.weight_update.weight_sync_schemes instead. This will be removed in a future version.
/usr

#### Loss module and optimizer

We build our loss as indicated in the dedicated tutorial, with its optimizer and target parameter updater:



In [15]:
from torch.optim import Adam
from torchrl.objectives import DQNLoss, SoftUpdate

loss = DQNLoss(value_network=policy, action_space=env.action_spec, delay_value=True)
optim = Adam(loss.parameters(), lr=0.02)
updater = SoftUpdate(loss, eps=0.99)


#### Logger

We’ll be using a CSV logger to log our results, and save rendered videos.

In [41]:
from torchrl._utils import logger as torchrl_logger
from torchrl.record import CSVLogger, VideoRecorder

path = "./training_loop"
logger = CSVLogger(exp_name="dqn", log_dir=path, video_format="mp4")
video_recorder = VideoRecorder(logger, tag="video")
record_env = TransformedEnv(
    GymEnv("CartPole-v1", from_pixels=True, pixels_only=False, max_episode_steps=1000), video_recorder
)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


#### Training loop

Instead of fixing a specific number of iterations to run, we will keep on training the network until it reaches a certain performance (arbitrarily defined as 200 steps in the environment – with CartPole, success is defined as having longer trajectories).



In [42]:
total_count = 0
total_episodes = 0
t0 = time.time()

for i, data in enumerate(collector):

    # Write data in replay buffer
    rb.extend(data)
    max_length = rb[:]["next", "step_count"].max()

    if len(rb) > init_rand_steps:

        # Optim loop (we do several optim steps
        # per batch collected for efficiency)
        for _ in range(optim_steps):
            sample = rb.sample(128)
            loss_vals = loss(sample)
            loss_vals["loss"].backward()
            optim.step()
            optim.zero_grad()

            #update exploration factor
            exploration_module.step(data.numel())

            #update target params
            updater.step()

            #logging
            if i % 10:
                torchrl_logger.info(f"Max num steps: {max_length}, rb length {len(rb)}")
            total_count += data.numel()
            total_episodes += data["next", "done"].sum()

    if max_length > 200:
        break

t1 = time.time()

torchrl_logger.info(
    f"solved after {total_count} steps, {total_episodes} episodes and in {t1 - t0}s."
)


2026-05-07 19:25:24,772 [torchrl][INFO]    solved after 1000 steps, 40 episodes and in 0.23181986808776855s. [END]


#### Rendering

Finally, we run the environment for as many steps as we can and save the video locally (notice that we are not exploring).



In [43]:
record_env.rollout(max_steps=1000, policy=policy)
video_recorder.dump()

In [44]:
from IPython.display import Video, display
display(Video('training_loop/dqn/videos/video_0.mp4', embed=True))